# Predicting Bug Resolution Time

## 0. Context

This notebook performs exploratory data analysis (EDA) on a synthetic bug-tracking dataset.

Goal:
- Understand distributions and relationships in the data
- Validate assumptions defined in data design
- Decide on target formulation for ML

Non-goals:
- Model training
- Feature engineering
- Pipeline implementation

## 1. Imports & Config
In this section will be imported all mandatory configs and predefined fields properties.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random
import os

# random.seed necessary to obtain an identical sample each time
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.options.display.float_format = '${:,.2f}'.format
pd.set_option('display.max_columns', 8)
pd.set_option('display.max_rows', 1000)

## 2. Load Dataset

In [4]:
df = pd.read_csv("../data/raw/bugs_synthetic.csv")
df.head()

,bug_id,created_at,closed_at,days_to_close,...,assignee_id,labels,summary,description
0,fakePrj-1,2025-06-13 02:14:20.333841,2025-07-05 04:43:41.333841,22,...,assignee_4,performance regression ui backend,fakesummary,interim1
1,fakePrj-2,2024-08-23 06:35:39.333841,2024-10-15 04:29:22.333841,52,...,assignee_5,backend ui regression,fakesummary,interim1
2,fakePrj-3,2025-04-01 17:51:46.333841,2025-04-29 11:36:25.333841,27,...,assignee_3,backend,fakesummary,interim1
3,fakePrj-4,2024-07-28 20:17:10.333841,2024-08-22 00:17:13.333841,24,...,assignee_3,ui backend regression,fakesummary,interim1
4,fakePrj-5,2024-07-17 00:28:40.333841,2024-10-07 04:11:38.333841,82,...,assignee_3,regression ui backend,fakesummary,interim1


### checking that all data being loaded

In [ ]:
#check that all of the fields has data, if 0 - then OK.
print(df.isna().sum())

bug_id           0
created_at       0
closed_at        0
days_to_close    0
priority         0
severity         0
component        0
assignee_id      0
labels           0
summary          0
description      0
dtype: int64


## 3. Dataset Overview
High-level inspection of dataset structure, size and column types.

In [18]:
#checking amount of cells and rows
df.shape

(1000, 11)

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['closed_at'] = pd.to_datetime(df['closed_at'], errors='coerce')

In [19]:
# checking of datatypes
df.dtypes

bug_id                   object
created_at       datetime64[ns]
closed_at        datetime64[ns]
days_to_close             int64
priority                 object
severity                 object
component                object
assignee_id              object
labels                   object
summary                  object
description              object
dtype: object

In [21]:
# code analysis
df.describe(include="all")

,bug_id,created_at,closed_at,days_to_close,...,assignee_id,labels,summary,description
count,1000,1000,1000,"$1,000.00",...,1000,1000,1000,1000
unique,1000,NaN,NaN,NaN,...,5,64,1,1
top,fakePrj-1,NaN,NaN,NaN,...,assignee_3,performance,fakesummary,interim1
freq,1,NaN,NaN,NaN,...,221,73,1000,1000
mean,NaN,2025-01-08 15:20:04.949840640,2025-02-24 17:42:51.387841024,$46.61,...,NaN,NaN,NaN,NaN
min,NaN,2024-01-09 23:27:37.333841,2024-01-16 22:13:00.333841,$0.00,...,NaN,NaN,NaN,NaN
25%,NaN,2024-07-09 06:14:40.833840896,2024-08-26 23:38:57.333840896,$25.00,...,NaN,NaN,NaN,NaN
50%,NaN,2025-01-10 08:18:02.333840896,2025-02-23 21:22:53.833840896,$47.00,...,NaN,NaN,NaN,NaN
75%,NaN,2025-07-13 07:54:21.833840896,2025-08-31 00:50:08.583840768,$70.00,...,NaN,NaN,NaN,NaN
max,NaN,2026-01-07 21:28:42.333841,2026-03-20 10:57:57.333841,$89.00,...,NaN,NaN,NaN,NaN
